In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error
import os

def train_quantile_models(train_df, features, target_col):
    """
    Trains two LightGBM models using quantile regression.
    Instead of predicting a single average, this outputs confidence intervals
    required for risk-adjusted quoting.

    Parameters
    ----------
    train_df : pandas.DataFrame
        The pre-processed training dataset.
    features : list
        List of feature column names used for training.
    target_col : str
        The column name of the actual transit days.

    Returns
    -------
    model_p50 : lightgbm.LGBMRegressor
        Model predicting the median expected transit time (50% confidence).
    model_p90 : lightgbm.LGBMRegressor
        Model predicting the conservative transit time (90% confidence).
    """
    print("Training LightGBM Quantile Models...")
    X_train = train_df[features]
    y_train = train_df[target_col]

    # P50 Model (Median Target Transit)
    print(" - Fitting P50 Model (Median Expectation)...")
    model_p50 = lgb.LGBMRegressor(
        objective='quantile',
        alpha=0.5,
        n_estimators=100,
        random_state=42
    )
    model_p50.fit(X_train, y_train)

    # P90 Model (Safe Conservatism Target)
    print(" - Fitting P90 Model (90% Confidence Buffer)...")
    model_p90 = lgb.LGBMRegressor(
        objective='quantile',
        alpha=0.9,
        n_estimators=100,
        random_state=42
    )
    model_p90.fit(X_train, y_train)

    return model_p50, model_p90

def evaluate_models(test_df, features, target_col, model_p50, model_p90):
    """
    Generates predictions on the unseen test set and evaluates basic accuracy.

    Parameters
    ----------
    test_df : pandas.DataFrame
        The chronological test dataset.
    features : list
        List of feature column names.
    target_col : str
        The column name of the actual transit days.
    model_p50, model_p90 : lightgbm.LGBMRegressor
        The trained models.

    Returns
    -------
    test_df : pandas.DataFrame
        The test dataframe appended with 'Pred_P50_Days' and 'Pred_P90_Days'.
    """
    print("Generating predictions on the test set...")
    X_test = test_df[features]
    y_test = test_df[target_col]

    test_df['Pred_P50_Days'] = model_p50.predict(X_test)
    test_df['Pred_P90_Days'] = model_p90.predict(X_test)

    # Basic Evaluation Metric (Mean Absolute Error on the P50 median prediction)
    mae_p50 = mean_absolute_error(y_test, test_df['Pred_P50_Days'])

    print("\n=== PHASE 3 VALIDATION RESULTS ===")
    print(f"Test Set Size: {len(test_df)} shipments")
    print(f"P50 Model Mean Absolute Error (MAE): {mae_p50:.2f} days")
    print("Note: The P90 model is evaluated on business metrics (NSL protection) in Phase 4.")
    print("==================================\n")

    return test_df


# --- Execution Block ---
if __name__ == "__main__":
    # Define file paths
    train_path = os.path.join("../..", "Data", "Processed", "train_data.csv")
    test_path = os.path.join("../..", "Data", "Processed", "test_data.csv")
    output_path = os.path.join("../..", "Data", "Processed", "Phase3_Predictions.csv")

    try:
        # 1. Load Data
        print(f"Loading split datasets...")
        train_data = pd.read_csv(train_path)
        test_data = pd.read_csv(test_path)

        actual_col = 'Calculated_Actual_Days' if 'Calculated_Actual_Days' in train_data.columns else 'Actual_Transit_Days'

        # 2. Define Features dynamically based on what was preprocessed
        feature_cols = [
            'Pickup_Day_of_Week', 'Pickup_Month', 'Is_Weekend_Pickup',
            'orig_cntry_cd', 'dest_cntry_cd', 'dest_pstl_cd',
            'Lane_Historical_Avg', 'Postal_Rolling_7D_Delay_Rate'
        ]

        # Safely include Cluster_ID if Phase 1.5 was merged in earlier
        if 'Cluster_ID' in train_data.columns:
            feature_cols.append('Cluster_ID')

        # 3. Model Training
        m_p50, m_p90 = train_quantile_models(train_data, feature_cols, actual_col)

        # 4. Validation & Output
        results_df = evaluate_models(test_data, feature_cols, actual_col, m_p50, m_p90)

        # 5. Save final predictions
        results_df.to_csv(output_path, index=False)
        print(f"Test predictions successfully saved to {output_path} for Phase 4 processing.")

    except FileNotFoundError:
        print("[ERROR] Could not find train_data.csv or test_data.csv.")
        print("Ensure you run 'phase3_preprocessing.py' first to generate these files.")

Loading split datasets...
Training LightGBM Quantile Models...
 - Fitting P50 Model (Median Expectation)...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000104 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 566
[LightGBM] [Info] Number of data points in the train set: 5824, number of used features: 7
[LightGBM] [Info] Start training from score 7.000000
 - Fitting P90 Model (90% Confidence Buffer)...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000033 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 566
[LightGBM] [Info] Number of data points in the train set: 5824, number of used features: 7
[LightGBM] [Info] Start training from score 12.000000
Generating predictions on the test se